# Member 1 — Exact CVRP Method | Version 1: Setup and Validation
**Goal:** establish the shared data format, integer distance matrix, and independent route validator.

**Git commit:** `feat(exact): add shared CVRP instance and feasibility validator`

This first version uses a reproducible synthetic instance so it runs immediately in Colab without external downloads. For the final report, replace the instance generator with the agreed CVRPLIB data parser and document the source. Both members must use identical instance data.

## 1. Environment and shared instance

In [ ]:
!pip -q install numpy pandas matplotlib ortools

In [ ]:
import math, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_CUSTOMERS = 20   # Increase to 31 to match the scale of the sample report
K = 5
CAPACITY = 100
rng = np.random.default_rng(SEED)

# Shared deterministic synthetic instance. Both members must use these same parameters.
coords = [(50.0, 50.0)] + [(float(x), float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
demands = [0] + [int(x) for x in rng.integers(4, 11, size=N_CUSTOMERS)]
n = len(coords)
dist = np.zeros((n,n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            dist[i,j] = int(math.floor(math.dist(coords[i], coords[j]) + 0.5))

print(f"Nodes: {n} (including depot 0); customers: {N_CUSTOMERS}; vehicles: {K}; capacity: {CAPACITY}")
print(f"Total demand: {sum(demands)}; total fleet capacity: {K*CAPACITY}")
assert sum(demands) <= K*CAPACITY, "Infeasible generated instance; adjust seed/demands."

## 2. Independent route validator

In [ ]:
def route_distance(route):
    """Distance for a customer list; depot 0 is added at both ends."""
    seq = [0] + list(route) + [0]
    return int(sum(dist[seq[i], seq[i+1]] for i in range(len(seq)-1)))

def solution_cost(routes):
    return sum(route_distance(r) for r in routes)

def validate_routes(routes, verbose=True):
    errors = []
    seen = [c for route in routes for c in route]
    if sorted(seen) != list(range(1,n)):
        errors.append("Every customer must appear exactly once.")
    if len(routes) > K:
        errors.append(f"Used {len(routes)} routes, exceeding fleet limit {K}.")
    for ridx, route in enumerate(routes, start=1):
        load = sum(demands[c] for c in route)
        if load > CAPACITY:
            errors.append(f"Route {ridx} load {load} exceeds capacity {CAPACITY}.")
        if not route:
            errors.append(f"Route {ridx} is empty.")
    cost = solution_cost(routes)
    if verbose:
        print("Feasible:", not errors, "| Cost:", cost, "| Routes:", len(routes))
        for e in errors: print(" -", e)
    return {"feasible": not errors, "cost": cost, "routes_used": len(routes), "errors": errors}

## 3. Smoke test

In [ ]:
# A simple feasible test partition. It is only a validator test, not an optimization result.
test_routes = [list(range(1,5)), list(range(5,9)), list(range(9,13)),
               list(range(13,17)), list(range(17,n))]
test_routes = [r for r in test_routes if r]
validate_routes(test_routes)

## Version 1 completion checklist
- [ ] Confirm total demand does not exceed fleet capacity.
- [ ] Confirm both members use the same coordinates, demands, distance rounding, `K`, and capacity.
- [ ] Commit this notebook before implementing the solver.
- [ ] Keep generated results out of the repository unless intentionally versioned.